In [37]:
kernel.silent(true)

# Logistic Regression — Scala 3

Predicting if a mouse is obese based on weight.

Formula: `σ(x) = 1 / (1 + e^(-(β₁x + β₀)))`

---

## 1. The Sigmoid

Takes any number and puts it between 0 and 1.

- Big negative → close to 0%
- Zero → exactly 50%
- Big positive → close to 100%

In [43]:
import scala.math.{exp, log}

def sigmoid(z: Double): Double = 1.0 / (1.0 + exp(-z))

println(f"sigmoid(-5) = ${sigmoid(-5)}%.4f  ← close to 0%%")
println(f"sigmoid( 0) = ${sigmoid(0)}%.4f   ← exactly 50%%")
println(f"sigmoid( 5) = ${sigmoid(5)}%.4f  ← close to 100%%")

sigmoid(-5) = 0.0067  ← close to 0%
sigmoid( 0) = 0.5000   ← exactly 50%
sigmoid( 5) = 0.9933  ← close to 100%
sigmoid( 0) = 0.5000   ← exactly 50%
sigmoid( 5) = 0.9933  ← close to 100%


---

## 2. Dataset (the answer key)

9 mice. Each row: weight + obese or not.

In [44]:
case class Mouse(weight: Double, obese: Int)

val data = List(
  Mouse(1.5, 0), Mouse(2.0, 0), Mouse(2.5, 0), Mouse(3.0, 0),
  Mouse(3.5, 1), Mouse(4.0, 1), Mouse(4.5, 1), Mouse(5.0, 1), Mouse(5.5, 1)
)

data.foreach { m =>
  println(f"  weight: ${m.weight}%.1fg → ${if (m.obese == 1) "obese" else "not obese"}")
}

  weight: 1.5g → not obese
  weight: 2.0g → not obese
  weight: 2.5g → not obese
  weight: 3.0g → not obese
  weight: 3.5g → obese
  weight: 4.0g → obese
  weight: 4.5g → obese
  weight: 5.0g → obese
  weight: 5.5g → obese
  weight: 1.5g → not obese
  weight: 2.0g → not obese
  weight: 2.5g → not obese
  weight: 3.0g → not obese
  weight: 3.5g → obese
  weight: 4.0g → obese
  weight: 4.5g → obese
  weight: 5.0g → obese
  weight: 5.5g → obese


---

## 3. The Loop — Gradient Descent

- **Step 0:** Random start (only once)
- **Step 1:** Grade (calculate likelihood)
- **Step 2:** Change β (move in the direction that improves the grade)
- **Repeat 1→2** until grade stops improving

**β₁** = steepness of the curve (how much weight matters)  
**β₀** = shifts the curve left/right (where the transition starts)  
**lr** = learning rate — size of each step (0.1 = move 10% of the gradient)  

The **gradient** tells the model which direction to move each β to improve.  
`grad = correct - guess` — this is the derivative that drives the adjustment.

In [45]:
var b1 = 0.0  // β₁
var b0 = 0.0  // β₀
val lr = 0.1  // learning rate

for (epoch <- 1 to 100000) {
  var gb1 = 0.0  // gradient for β₁
  var gb0 = 0.0  // gradient for β₀
  var lh = 1.0   // likelihood accumulator
  for (m <- data) {
    val p = sigmoid(b1 * m.weight + b0)  // model's guess
    val grad = m.obese - p               // derivative of log-likelihood
    gb1 += grad * m.weight               // gradient for β₁
    gb0 += grad                          // gradient for β₀
    lh *= (if (m.obese == 1) p else 1.0 - p)  // score for correct answer
  }
  b1 += lr * gb1 / data.size  // adjust β₁ to maximize likelihood
  b0 += lr * gb0 / data.size  // adjust β₀ to maximize likelihood

  if (epoch == 1 || epoch == 10 || epoch == 100 || epoch == 500 || epoch == 2000 || epoch == 10000 || epoch == 100000)
    println(f"  Iteration $epoch%4d → β₁=$b1%+.4f  β₀=$b0%+.4f  L=$lh%.6f")
}
// ↑ L goes UP = model is improving

// b1 and b0 ARE the trained model — these two numbers are all you need
println(f"\n✅ Trained model: β₁=$b1%.4f (steepness)  β₀=$b0%.4f (shift)")

  Iteration    1 → β₁=+0.0750  β₀=+0.0056  L=0.001953
  Iteration   10 → β₁=+0.2365  β₀=-0.0740  L=0.004473
  Iteration  100 → β₁=+0.5099  β₀=-1.0777  L=0.013322
  Iteration  500 → β₁=+1.2736  β₀=-3.7911  L=0.086972
  Iteration 2000 → β₁=+2.4637  β₀=-7.8100  L=0.274717
  Iteration 10000 → β₁=+4.5904  β₀=-14.8037  L=0.535651
  Iteration 100000 → β₁=+10.4019  β₀=-33.7297  L=0.865519

✅ Trained model: β₁=10.4019 (steepness)  β₀=-33.7297 (shift)
  Iteration    1 → β₁=+0.0750  β₀=+0.0056  L=0.001953
  Iteration   10 → β₁=+0.2365  β₀=-0.0740  L=0.004473
  Iteration  100 → β₁=+0.5099  β₀=-1.0777  L=0.013322
  Iteration  500 → β₁=+1.2736  β₀=-3.7911  L=0.086972
  Iteration 2000 → β₁=+2.4637  β₀=-7.8100  L=0.274717
  Iteration 10000 → β₁=+4.5904  β₀=-14.8037  L=0.535651
  Iteration 100000 → β₁=+10.4019  β₀=-33.7297  L=0.865519

✅ Trained model: β₁=10.4019 (steepness)  β₀=-33.7297 (shift)


---

## 4. Predictions

The trained model = **b1** and **b0** from step 3.  
We use `sigmoid(b1 * weight + b0)` to predict any new mouse.  
Above 50% → obese.

In [46]:
val threshold = 0.5  // decision boundary — change to 0.8 for stricter

def predictAndShow(weight: Double): Unit =
  val prob = sigmoid(b1 * weight + b0)
  val label = if (prob > threshold) "OBESE" else "NOT OBESE"
  println(f"  $weight%.1fg → ${prob * 100}%5.1f%% → $label")

// New mice (never seen)l
predictAndShow(1.0)
predictAndShow(2.8)
predictAndShow(3.3)
predictAndShow(4.2)
predictAndShow(6.0)

// Run on training data
println()
data.foreach(m => predictAndShow(m.weight))

  1.0g →   0.0% → NOT OBESE
  2.8g →   1.0% → NOT OBESE
  3.3g →  64.5% → OBESE
  4.2g → 100.0% → OBESE
  6.0g → 100.0% → OBESE

  1.5g →   0.0% → NOT OBESE
  2.0g →   0.0% → NOT OBESE
  2.5g →   0.0% → NOT OBESE
  3.0g →   7.4% → NOT OBESE
  3.5g →  93.6% → OBESE
  4.0g → 100.0% → OBESE
  4.5g → 100.0% → OBESE
  5.0g → 100.0% → OBESE
  5.5g → 100.0% → OBESE
